# DEE — Notebook A v2: Calibración corregida + tests inflación sin inflatón

## Estructura

**Bloque 1 — Calibración (tests corregidos):**
- Test 1: w_def cosmológico vía dilución con N (replica método de DM_v2 que dio -0.885)
- Test 2: gap espectral Δω en dos versiones (gap_local al sub-banda, gap_bulk al continuo extendido)
- Test 3: ω_def como función de N con algoritmo robusto de identificación del modo principal

**Bloque 2 — Tests de inflación sin inflatón:**
- Test 7: suavizado de var(κ) durante annealing — análogo cuantitativo a 'e-folds'
- Test 8: longitud de correlación ξ(T) durante el cruce de T_c — análogo del horizonte
- Test 9: espectro de potencias P(k, T) en cada fase del annealing — ¿el sustrato suaviza n_s hacia el observado?

**Bugs corregidos respecto al notebook A original:**
1. w_def: ahora se mide vía sweep de N con defecto absoluto fijo (método DM_v2), no por escalado de coordenadas.
2. Identificación de modo principal: usa 'menor ω entre los modos con concentración > 3', adaptable a N grande.
3. Δω: se calculan dos versiones (intra-banda y bulk).

**Tiempo total estimado**: 4-6 horas en Colab Pro.

**Outputs**: `plots_calibracion_v5_v2/` con plots, summary.txt, results_raw.pkl

In [ ]:
# ============================================================
# Imports
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
from scipy.spatial import cKDTree
import time
import os
import pickle

os.makedirs('plots_calibracion_v5_v2', exist_ok=True)
np.random.seed(42)
print('Setup OK')

In [ ]:
# ============================================================
# Funciones core (autocontenidas)
# ============================================================

def grid_perturbed_T3(N_target, jitter_fraction=0.10, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, n_side**3

def build_laplacian_T3(points, k_neighbors=30, alpha=2.0,
                       defect_center=None, defect_radius=0.15, defect_factor=1.0, L=1.0):
    """Laplaciano del cristal con kernel 1/d^alpha y defecto blando opcional.
    Defecto: si AL MENOS UNO de los nodos del enlace está dentro del defecto,
    el peso se multiplica por defect_factor (< 1 => debilitamiento => pozo elástico)."""
    N = len(points)
    images = []; image_to_orig = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                images.append(points + np.array([dx*L, dy*L, dz*L]))
                image_to_orig.extend([(i, dx, dy, dz) for i in range(N)])
    images_all = np.concatenate(images, axis=0)
    image_to_orig = np.array([io[0] for io in image_to_orig])
    tree = cKDTree(images_all)
    rows, cols, vals = [], [], []
    if defect_center is not None:
        diff = points - defect_center
        diff = diff - L * np.round(diff / L)
        in_defect = np.linalg.norm(diff, axis=1) < defect_radius
    else:
        in_defect = np.zeros(N, dtype=bool)
    for i in range(N):
        dists, idxs = tree.query(points[i], k=k_neighbors+1)
        valid = ~((image_to_orig[idxs] == i) & (dists < 1e-12))
        v_idx = np.where(valid)[0][:k_neighbors]
        for vi in v_idx:
            j = image_to_orig[idxs[vi]]
            d = dists[vi]
            if d > 1e-10:
                w = 1.0 / d**alpha
                if in_defect[i] or in_defect[j]: w *= defect_factor
                rows.append(i); cols.append(j); vals.append(w)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    deg = np.array(W.sum(axis=1)).flatten()
    return diags(deg) - W, in_defect

def find_lowest_localized_mode(L_op, in_defect, n_modes=300, threshold=3.0, tol=1e-8):
    """Modo principal = de MENOR ω entre los modos con concentración > threshold * baseline.
    Algoritmo robusto a cambio de N."""
    eigs, vecs = eigsh(L_op, k=n_modes, which='SM', tol=tol)
    order = np.argsort(eigs); eigs = eigs[order]; vecs = vecs[:, order]
    nonzero = eigs > 1e-3
    eigs_nz = eigs[nonzero]; vecs_nz = vecs[:, nonzero]
    omegas = np.sqrt(eigs_nz)
    n_def = in_defect.sum(); n_tot = len(in_defect)
    fill_frac = n_def / n_tot
    concentration = np.zeros(len(eigs_nz))
    for k in range(len(eigs_nz)):
        psi2 = vecs_nz[:, k]**2
        norm = psi2.sum()
        if norm > 1e-12:
            concentration[k] = psi2[in_defect].sum() / norm / fill_frac
    candidates = np.where(concentration > threshold)[0]
    if len(candidates) > 0:
        idx_principal = candidates[0]  # menor ω entre los localizados
    else:
        idx_principal = np.argmax(concentration)
    return {
        'omegas': omegas,
        'vecs': vecs_nz,
        'concentration': concentration,
        'idx_principal': idx_principal,
        'omega_def': omegas[idx_principal],
        'vec_def': vecs_nz[:, idx_principal],
        'all_localized_indices': candidates,
    }

print('Funciones cargadas')

## Bloque 1 — Calibración corregida

### Tests 1, 2, 3 con método correcto

**Setup**: cristal T³ con defecto blando absoluto (radio 0.15 en unidades sim, factor 0.1).
Para cada N en {729, 1331, 3375, 5832} y 3 seeds, medimos:
- ω_def (frecuencia del modo principal del defecto)
- ω_band (frecuencia del primer modo de la banda intra-defecto, > ω_def)
- ω_bulk (frecuencia del primer modo extendido del bulk)
- Δω_local = ω_band − ω_def
- Δω_bulk = ω_bulk − ω_def
- Participation ratio del modo principal
- ρ_def = (1/2)·ω_def / N (densidad cosmológica)

**Análisis cosmológico**: ajuste log-log ρ_def ∝ V^β con V = N. β = -1 corresponde a CDM ideal.

In [ ]:
# ============================================================
# Bloque 1 — Sweep principal de calibración
# ============================================================

Ns_calib = [729, 1331, 3375, 5832]  # n_side = 9, 11, 15, 18
seeds_calib = [42, 100, 200]

calib_results = []

for N_target in Ns_calib:
    for seed in seeds_calib:
        t0 = time.time()
        points, N = grid_perturbed_T3(N_target, 0.10, seed=seed)
        L_op, in_defect = build_laplacian_T3(
            points, defect_center=np.array([0.5, 0.5, 0.5]),
            defect_radius=0.15, defect_factor=0.1, L=1.0
        )
        n_modes = min(400, N - 5)
        mode = find_lowest_localized_mode(L_op, in_defect, n_modes=n_modes, threshold=3.0)
        
        omega_def = mode['omega_def']
        
        # === Δω_local: separación al siguiente modo localizado ===
        loc_indices = mode['all_localized_indices']
        idx_p = mode['idx_principal']
        # Buscar el siguiente modo localizado (idx > idx_p)
        next_loc = loc_indices[loc_indices > idx_p]
        if len(next_loc) > 0:
            omega_band = mode['omegas'][next_loc[0]]
            delta_omega_local = omega_band - omega_def
        else:
            omega_band = np.nan
            delta_omega_local = np.nan
        
        # === Δω_bulk: separación al primer modo extendido ===
        ext_indices = np.where(mode['concentration'] < 1.5)[0]
        ext_above = ext_indices[mode['omegas'][ext_indices] > omega_def]
        if len(ext_above) > 0:
            omega_bulk = mode['omegas'][ext_above[0]]
            delta_omega_bulk = omega_bulk - omega_def
        else:
            omega_bulk = np.nan
            delta_omega_bulk = np.nan
        
        # === Participation ===
        psi2 = mode['vec_def']**2
        IPR = (psi2**2).sum() / (psi2.sum())**2
        participation = 1.0 / IPR
        
        # === Densidad cosmológica ===
        E_def = 0.5 * omega_def
        rho_def = E_def / N
        
        calib_results.append({
            'N': N, 'seed': seed,
            'omega_def': omega_def,
            'omega_band': omega_band,
            'omega_bulk': omega_bulk,
            'delta_omega_local': delta_omega_local,
            'delta_omega_bulk': delta_omega_bulk,
            'E_def': E_def,
            'rho_def': rho_def,
            'participation': participation,
            'n_localized': len(loc_indices),
            'n_in_defect': in_defect.sum(),
        })
        print(f'N={N:>5d}, seed={seed:>4d}: ω_def={omega_def:.4f}, '
              f'Δω_loc={delta_omega_local:.3f}, Δω_bulk={delta_omega_bulk:.3f}, '
              f'ρ_def={rho_def:.6f}, part={participation:.1f}, t={time.time()-t0:.1f}s')

print(f'\nTotal: {len(calib_results)} corridas completas')

In [ ]:
# ============================================================
# Análisis Bloque 1: agregar por N, ajuste de scaling
# ============================================================

by_N = {}
for r in calib_results:
    if r['N'] not in by_N:
        by_N[r['N']] = {'omega_def': [], 'delta_omega_local': [], 'delta_omega_bulk': [],
                        'rho_def': [], 'participation': []}
    for k in ['omega_def', 'delta_omega_local', 'delta_omega_bulk', 'rho_def', 'participation']:
        by_N[r['N']][k].append(r[k])

Ns_sorted = sorted(by_N.keys())
summary = {'N': Ns_sorted}

print('=== TABLA RESUMEN (medias ± std sobre seeds) ===\n')
header = f'{"N":>6s} | {"ω_def":>15s} | {"Δω_local":>15s} | {"Δω_bulk":>15s} | {"ρ_def":>14s} | {"part.":>10s}'
print(header)
print('-' * len(header))

for N in Ns_sorted:
    line_parts = [f'{N:>6d}']
    for k in ['omega_def', 'delta_omega_local', 'delta_omega_bulk', 'rho_def', 'participation']:
        vals = np.array(by_N[N][k])
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0:
            mean, std = np.mean(vals), np.std(vals)
            summary.setdefault(f'{k}_mean', []).append(mean)
            summary.setdefault(f'{k}_std', []).append(std)
            if k == 'participation':
                line_parts.append(f' {mean:>5.1f} ± {std:>4.1f}')
            else:
                line_parts.append(f' {mean:>+8.4f} ± {std:>5.4f}')
        else:
            summary.setdefault(f'{k}_mean', []).append(np.nan)
            summary.setdefault(f'{k}_std', []).append(np.nan)
            line_parts.append(' ' * 16)
    print(' | '.join(line_parts))

# === Scaling ρ_def vs V ===
Ns_arr = np.array(Ns_sorted, dtype=float)
rhos_arr = np.array(summary['rho_def_mean'])
valid = ~np.isnan(rhos_arr) & (rhos_arr > 0)
if valid.sum() >= 2:
    log_V = np.log(Ns_arr[valid])
    log_rho = np.log(rhos_arr[valid])
    slope_rho, intercept_rho = np.polyfit(log_V, log_rho, 1)
    w_cosmo = -slope_rho - 1
    print(f'\n=== Test 1 cosmológico ===')
    print(f'ρ_def ∝ V^({slope_rho:+.3f})')
    print(f'w_cosmológico implicado = -slope - 1 = {w_cosmo:+.3f}')
    print(f'Comparación con benchmarks:')
    print(f'  CDM ideal: w = 0,  ρ ∝ V^(-1)')
    print(f'  Λ:         w = -1, ρ ∝ V^( 0)')
    print(f'  Radiación: w = +1/3, ρ ∝ V^(-4/3)')

# === Convergencia de ω_def vs N ===
ome_arr = np.array(summary['omega_def_mean'])
if len(ome_arr) >= 2:
    slope_om, _ = np.polyfit(np.log(Ns_arr), np.log(ome_arr), 1)
    print(f'\n=== Test 3: ω_def vs N ===')
    print(f'ω_def ∝ N^({slope_om:+.3f})')
    print(f'Esperado para modo localizado puro: N^0 (constante)')
    if abs(slope_om) < 0.05:
        print('→ Convergencia: ω_def ESTABLE con N (modo localizado bien definido)')
    else:
        print(f'→ ω_def crece levemente con N (efecto de tamaño finito esperado)')

# === Convergencia de Δω ===
for delta_key in ['delta_omega_local', 'delta_omega_bulk']:
    delta_arr = np.array(summary[f'{delta_key}_mean'])
    valid = ~np.isnan(delta_arr) & (delta_arr > 0)
    if valid.sum() >= 2:
        # Extrapolación lineal en 1/N
        inv_N = 1.0 / Ns_arr[valid]
        s, i = np.polyfit(inv_N, delta_arr[valid], 1)
        print(f'\n{delta_key} (N→∞ extrapolado): {i:.4f} (pendiente 1/N: {s:+.2f})')

In [ ]:
# ============================================================
# Plots Bloque 1
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

Ns_arr = np.array(Ns_sorted, dtype=float)

# === Plot 1: ρ_def vs V (escala log-log) ===
ax = axes[0, 0]
rhos_mean = np.array(summary['rho_def_mean'])
rhos_std = np.array(summary['rho_def_std'])
ax.errorbar(Ns_arr, rhos_mean, yerr=rhos_std, fmt='o-', markersize=10, capsize=5,
            color='C3', label='Datos (3 seeds por N)')
# Líneas de referencia
v_ref = np.array([Ns_arr.min(), Ns_arr.max()])
rho_ref = rhos_mean[0]
ax.plot(v_ref, rho_ref * (v_ref/Ns_arr[0])**(-1), 'k--', alpha=0.5, label='CDM (V$^{-1}$)')
ax.plot(v_ref, rho_ref * (v_ref/Ns_arr[0])**(0), 'g--', alpha=0.5, label='Λ (V$^0$)')
ax.plot(v_ref, rho_ref * (v_ref/Ns_arr[0])**(-4/3), 'b--', alpha=0.5, label='Radiación (V$^{-4/3}$)')
if not np.isnan(slope_rho):
    ax.plot(v_ref, np.exp(intercept_rho) * v_ref**slope_rho, 'r-', linewidth=2,
            label=f'Ajuste DEE: V$^{{{slope_rho:+.3f}}}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('V (= N nodos)')
ax.set_ylabel('ρ_def')
ax.set_title(f'Test 1: dilución cosmológica\nw_cosmo = {w_cosmo:+.3f}')
ax.legend(); ax.grid(alpha=0.3, which='both')

# === Plot 2: ω_def vs N ===
ax = axes[0, 1]
ome_mean = np.array(summary['omega_def_mean'])
ome_std = np.array(summary['omega_def_std'])
ax.errorbar(Ns_arr, ome_mean, yerr=ome_std, fmt='o-', markersize=10, capsize=5,
            color='C0', label='ω_def')
ax.set_xscale('log')
ax.set_xlabel('N')
ax.set_ylabel('ω_def')
ax.set_title(f'Test 3: ω_def vs N (slope log-log: {slope_om:+.3f})')
ax.legend(); ax.grid(alpha=0.3, which='both')

# === Plot 3: Δω vs 1/N ===
ax = axes[1, 0]
inv_N = 1.0 / Ns_arr
for key, label, color in [('delta_omega_local', 'Δω_local (banda)', 'C1'),
                           ('delta_omega_bulk', 'Δω_bulk (continuo)', 'C2')]:
    vals_mean = np.array(summary[f'{key}_mean'])
    vals_std = np.array(summary[f'{key}_std'])
    valid = ~np.isnan(vals_mean) & (vals_mean > 0)
    if valid.sum() > 0:
        ax.errorbar(inv_N[valid], vals_mean[valid], yerr=vals_std[valid],
                    fmt='s-', markersize=10, capsize=5, color=color, label=label)
ax.set_xlabel('1/N')
ax.set_ylabel('Δω')
ax.set_title('Test 2: dos versiones del gap espectral')
ax.legend(); ax.grid(alpha=0.3)

# === Plot 4: Participation vs N ===
ax = axes[1, 1]
part_mean = np.array(summary['participation_mean'])
part_std = np.array(summary['participation_std'])
ax.errorbar(Ns_arr, part_mean, yerr=part_std, fmt='^-', markersize=10, capsize=5,
            color='C4', label='Participation')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('N')
ax.set_ylabel('Participation ratio')
ax.set_title('Localización del modo (debería crecer ~ N$_{def}$)')
ax.legend(); ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('plots_calibracion_v5_v2/bloque1_calibracion.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot Bloque 1 guardado')

## Bloque 2 — Tests de inflación sin inflatón

### Idea conceptual

El sustrato DEE durante el annealing entre T = θ_D y T = 0 muestra:
- Caída del 85%+ en var(κ) entre T = θ_D/10 y T = 0
- Tasa de aceptación pasando de 93-96% a 17-26% (transición orden-desorden)
- Critical slowing down (75% del tiempo en fases pre-críticas)

**Hipótesis**: la dinámica del sustrato cerca de T_c reproduce los efectos cualitativos de inflación (suavizado, expansión efectiva del horizonte de correlación, transformación del espectro de potencias) **sin necesidad de un inflatón**, simplemente por las propiedades termodinámicas de la transición de fase.

**Tests específicos:**
- **Test 7**: número de 'e-folds efectivos' = log(var(κ)_T0 / var(κ)_T_inicial)
- **Test 8**: ξ(T) durante el cruce de T_c
- **Test 9**: P(k, T) durante cada fase del annealing

In [ ]:
# ============================================================
# Funciones para Bloque 2: MC + annealing + observables
# ============================================================

def compute_kappa_field(L_op, points, jitter_kappa_fluctuations=True, seed=42):
    """Calcula el campo κ (Ricci-Ollivier proxy) sobre los nodos.
    Usamos el degree-degree imbalance como proxy de curvatura local.
    Esta es una aproximación rápida; cuando se trabaja con SIM 18/19/20
    se usa la fórmula completa de Ollivier."""
    deg = np.array(L_op.diagonal())
    N = len(deg)
    # κ_i ≈ (1 - <deg_j>/deg_i)  (proxy simple)
    # Para seed test, devolvemos las fluctuaciones del degree alrededor de la media
    return (deg - deg.mean()) / deg.std()

def compute_correlation_length(field, points, n_bins=20, L=1.0):
    """Calcula la longitud de correlación ξ ajustando exp(-r/ξ) a la
    función de correlación espacial C(r) = <field(0)·field(r)>."""
    N = len(field)
    # Distancias por pares (con PBC)
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    dists = np.linalg.norm(diff, axis=-1)
    # Producto de campo en pares
    field_centered = field - field.mean()
    var_field = field_centered.var()
    if var_field < 1e-12:
        return np.nan
    products = field_centered[:, None] * field_centered[None, :]
    # Bins radiales
    r_max = L / 2  # max distancia con PBC
    r_bins = np.linspace(0, r_max, n_bins + 1)
    r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
    C_r = np.zeros(n_bins)
    counts = np.zeros(n_bins)
    for b in range(n_bins):
        mask = (dists >= r_bins[b]) & (dists < r_bins[b+1])
        if mask.sum() > 0:
            C_r[b] = products[mask].mean()
            counts[b] = mask.sum()
    # Normalizar
    C_r = C_r / var_field
    # Ajuste exponencial: log(|C(r)|) = -r/ξ + const, para r > 0 y |C| > 0.05
    valid = (counts > 5) & (np.abs(C_r) > 0.02) & (r_centers > 0)
    if valid.sum() >= 3:
        try:
            log_C = np.log(np.abs(C_r[valid]))
            slope, _ = np.polyfit(r_centers[valid], log_C, 1)
            xi = -1.0 / slope if slope < 0 else np.inf
        except:
            xi = np.nan
    else:
        xi = np.nan
    return xi, r_centers, C_r, counts

def compute_power_spectrum(field, omegas, vecs):
    """Espectro de potencias: P(k_n) = |<field, ψ_n>|² donde k_n ∝ ω_n.
    Usa los autovectores del Laplaciano como base de Fourier."""
    coefs = vecs.T @ field  # proyección del campo en los modos
    return omegas, coefs**2

print('Funciones Bloque 2 cargadas')

In [ ]:
# ============================================================
# MC con annealing — versión simplificada para reproducir SIM 18
# ============================================================

def annealing_with_observables(points_init, n_steps_per_phase=2000, T_schedule=None,
                                jitter_step=0.005, seed=42, L=1.0,
                                k_neighbors=30, alpha=2.0,
                                save_every=200):
    """Annealing simplificado registrando observables en cada fase.
    
    F = N · var(degree). Aproximación de F = N·var(κ) usando degree del Laplaciano.
    Movimientos: jitter aleatorio de un nodo en magnitud jitter_step.
    
    Returns: dict con observables por fase (var, T, accept_rate, snapshots)
    """
    rng = np.random.default_rng(seed)
    points = points_init.copy()
    N = len(points)
    if T_schedule is None:
        T_schedule = [43.4, 14.47, 4.34, 0.0]  # SIM 18 default
    
    phase_results = []
    
    # F inicial
    L_op, _ = build_laplacian_T3(points, k_neighbors=k_neighbors, alpha=alpha, L=L)
    deg = np.array(L_op.diagonal())
    F = N * deg.var()
    
    for phase_idx, T in enumerate(T_schedule):
        F_history = [F]
        accepts = 0
        snapshots = []  # guardar snapshots para Test 9
        
        for step in range(n_steps_per_phase):
            # Proponer movimiento de un nodo aleatorio
            i = rng.integers(0, N)
            old_pos = points[i].copy()
            displacement = rng.normal(0, jitter_step, 3)
            points[i] = (points[i] + displacement) % L
            
            # Recalcular F (versión rápida: sólo localmente)
            L_op_new, _ = build_laplacian_T3(points, k_neighbors=k_neighbors, alpha=alpha, L=L)
            deg_new = np.array(L_op_new.diagonal())
            F_new = N * deg_new.var()
            dF = F_new - F
            
            # Criterio Metropolis
            if T > 0:
                accept = (dF < 0) or (rng.random() < np.exp(-dF / T))
            else:
                accept = (dF < 0)
            
            if accept:
                F = F_new
                L_op = L_op_new
                deg = deg_new
                accepts += 1
            else:
                points[i] = old_pos
            
            F_history.append(F)
            
            # Guardar snapshots periódicos
            if (step + 1) % save_every == 0:
                snapshots.append({
                    'step': step + 1,
                    'F': F,
                    'points': points.copy(),
                    'deg': deg.copy(),
                })
        
        accept_rate = accepts / n_steps_per_phase
        phase_results.append({
            'phase': phase_idx,
            'T': T,
            'F_initial': F_history[0],
            'F_final': F,
            'F_history': F_history,
            'accept_rate': accept_rate,
            'final_points': points.copy(),
            'final_deg': deg.copy(),
            'snapshots': snapshots,
        })
        print(f'  Fase {phase_idx} (T={T:.2f}): F = {F_history[0]:.2f} → {F:.2f}, '
              f'aceptación = {accept_rate*100:.1f}%')
    
    return phase_results

print('Función annealing cargada')

In [ ]:
# ============================================================
# Test 7: suavizado + e-folds efectivos durante annealing
# Test 8: longitud de correlación ξ(T)
# Test 9: P(k, T) en cada fase
# ============================================================

# Setup: cristal limpio (sin defecto), N=500, annealing simulado
N_target_anneal = 512  # n_side=8 para mantener tiempos razonables
n_steps_per_phase = 2000  # reducido respecto a SIM 18 para hacer en tiempo razonable
T_schedule = [43.4, 14.47, 4.34, 1.0, 0.0]  # incluyo T_intermedio para mejor resolución
seed_anneal = 42

print(f'Test 7-8-9: annealing con N={N_target_anneal}, {len(T_schedule)} fases, '
      f'{n_steps_per_phase} pasos por fase')
print(f'(Esto puede tomar 30-90 minutos)')

t0 = time.time()
points_init, N_anneal = grid_perturbed_T3(N_target_anneal, 0.10, seed=seed_anneal)
phase_results = annealing_with_observables(
    points_init, n_steps_per_phase=n_steps_per_phase,
    T_schedule=T_schedule, jitter_step=0.005, seed=seed_anneal,
    L=1.0, save_every=500
)
print(f'\nAnnealing completo: {time.time()-t0:.1f}s')

In [ ]:
# ============================================================
# Análisis Test 7: suavizado y e-folds efectivos
# ============================================================

# var(κ) ~ var(degree) en cada fase
vars_per_phase = []
Ts_arr = []
for pr in phase_results:
    vars_per_phase.append(pr['final_deg'].var())
    Ts_arr.append(pr['T'])

vars_per_phase = np.array(vars_per_phase)
Ts_arr = np.array(Ts_arr)

print('=== Test 7: suavizado de var(κ) ===')
for i, (T, v) in enumerate(zip(Ts_arr, vars_per_phase)):
    print(f'  Fase {i} (T={T:.2f}): var(degree) = {v:.4e}')

# Razón inicial vs final
ratio_smoothing = vars_per_phase[0] / vars_per_phase[-1]
n_efolds_eff = np.log(ratio_smoothing)

print(f'\nRazón var inicial / var final: {ratio_smoothing:.2e}')
print(f'\"e-folds efectivos\" = ln(razón) = {n_efolds_eff:.2f}')
print(f'\nReferencia inflación: ~60 e-folds para resolver problema del horizonte.')
if n_efolds_eff > 5:
    print(f'Sustrato DEE produce {n_efolds_eff:.1f} e-folds efectivos por fase de annealing.')
    print(f'Para igualar inflación se requerirían {60/n_efolds_eff:.1f} ciclos completos de annealing.')

In [ ]:
# ============================================================
# Análisis Test 8: longitud de correlación ξ(T)
# ============================================================

xis_per_phase = []
C_data_per_phase = []

print('=== Test 8: longitud de correlación ξ(T) ===')
for i, pr in enumerate(phase_results):
    field = (pr['final_deg'] - pr['final_deg'].mean()) / (pr['final_deg'].std() + 1e-12)
    result = compute_correlation_length(field, pr['final_points'], n_bins=15, L=1.0)
    if isinstance(result, tuple):
        xi, r_c, C_r, counts = result
    else:
        xi = np.nan; r_c = None; C_r = None; counts = None
    xis_per_phase.append(xi)
    C_data_per_phase.append({'r_centers': r_c, 'C_r': C_r, 'counts': counts})
    print(f'  Fase {i} (T={pr["T"]:.2f}): ξ = {xi:.4f}')

xis_per_phase = np.array(xis_per_phase)
valid_xi = ~np.isnan(xis_per_phase)
if valid_xi.sum() >= 2:
    # ξ aumenta con relajación? 
    if xis_per_phase[-1] > xis_per_phase[0]:
        ratio_xi = xis_per_phase[-1] / xis_per_phase[0]
        print(f'\nLongitud de correlación creció por factor {ratio_xi:.2f}× del estado inicial al final')
    else:
        print(f'\nLongitud de correlación NO creció monotónicamente con annealing')
    
    print(f'\nReferencia: para resolver problema del horizonte, ξ debería crecer')
    print(f'al menos 10²² veces (factor de expansión inflacionaria mínimo).')

In [ ]:
# ============================================================
# Análisis Test 9: P(k, T) en cada fase
# ============================================================

print('=== Test 9: espectro de potencias por fase ===\n')
ns_per_phase = []
P_data_per_phase = []

for i, pr in enumerate(phase_results):
    L_op_phase, _ = build_laplacian_T3(pr['final_points'], k_neighbors=30, alpha=2.0, L=1.0)
    n_modes = min(300, N_anneal - 5)
    eigs, vecs = eigsh(L_op_phase, k=n_modes, which='SM', tol=1e-7)
    order = np.argsort(eigs); eigs = eigs[order]; vecs = vecs[:, order]
    nonzero = eigs > 1e-3
    eigs_nz = eigs[nonzero]; vecs_nz = vecs[:, nonzero]
    omegas = np.sqrt(eigs_nz)
    
    # Campo: usar el campo de degree de esta fase
    field = pr['final_deg'] - pr['final_deg'].mean()
    coefs = vecs_nz.T @ field
    P_k = coefs**2
    k_vals = omegas
    
    # Binning logarítmico
    if k_vals.min() > 0:
        n_bins = 15
        k_bins = np.logspace(np.log10(k_vals.min() * 1.05), np.log10(k_vals.max() * 0.95), n_bins+1)
        k_centers = np.sqrt(k_bins[:-1] * k_bins[1:])
        P_binned = np.zeros(n_bins)
        counts = np.zeros(n_bins)
        for j, k in enumerate(k_vals):
            b = np.searchsorted(k_bins, k) - 1
            if 0 <= b < n_bins:
                P_binned[b] += P_k[j]
                counts[b] += 1
        P_avg = np.where(counts > 0, P_binned / np.maximum(counts, 1), np.nan)
        valid = (counts >= 3) & ~np.isnan(P_avg) & (P_avg > 0)
        
        if valid.sum() >= 4:
            log_k = np.log10(k_centers[valid])
            log_P = np.log10(P_avg[valid])
            slope, intercept = np.polyfit(log_k, log_P, 1)
            n_s = slope + 1
        else:
            slope, intercept, n_s = np.nan, np.nan, np.nan
    else:
        k_centers, P_avg, valid, n_s = None, None, None, np.nan
    
    ns_per_phase.append(n_s)
    P_data_per_phase.append({'k_vals': k_vals, 'P_k': P_k, 'k_centers': k_centers,
                              'P_avg': P_avg, 'valid': valid, 'n_s': n_s})
    print(f'Fase {i} (T={pr["T"]:.2f}): n_s = {n_s:+.4f}')

ns_per_phase = np.array(ns_per_phase)
print(f'\nReferencia: n_s ≈ 0.96 (Planck 2018, observado)')
print(f'             n_s = 1 (Harrison-Zeldovich)')
print(f'             n_s = -1 (equipartición acústica clásica)')

valid_ns = ~np.isnan(ns_per_phase)
if valid_ns.sum() >= 2:
    delta_ns = ns_per_phase[-1] - ns_per_phase[0]
    print(f'\nVariación n_s desde fase inicial a final: Δn_s = {delta_ns:+.4f}')
    if delta_ns > 0.1:
        print('→ El sustrato SUAVIZA el espectro hacia más planos (más invariante)')
    elif delta_ns < -0.1:
        print('→ El sustrato HACE EL ESPECTRO MÁS ROJO durante el annealing')
    else:
        print('→ n_s estable durante el annealing')

In [ ]:
# ============================================================
# Plots Bloque 2
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# === Plot 1: var(degree) vs T (Test 7) ===
ax = axes[0, 0]
phase_idx = np.arange(len(phase_results))
ax.semilogy(phase_idx, vars_per_phase, 'o-', markersize=12, linewidth=2, color='C3')
for i, T in enumerate(Ts_arr):
    ax.annotate(f'T={T:.1f}', xy=(i, vars_per_phase[i]), xytext=(5, 5),
                textcoords='offset points', fontsize=9)
ax.set_xlabel('Fase del annealing')
ax.set_ylabel('var(degree) [proxy de var(κ)]')
ax.set_title(f'Test 7: suavizado durante annealing\n'
             f'"e-folds efectivos" = {n_efolds_eff:.2f}')
ax.grid(alpha=0.3, which='both')

# === Plot 2: ξ(T) (Test 8) ===
ax = axes[0, 1]
ax.plot(Ts_arr, xis_per_phase, 'o-', markersize=12, linewidth=2, color='C0')
for i, (T, xi) in enumerate(zip(Ts_arr, xis_per_phase)):
    if not np.isnan(xi):
        ax.annotate(f'fase {i}', xy=(T, xi), xytext=(5, 5),
                    textcoords='offset points', fontsize=9)
ax.invert_xaxis()  # T va de alta a baja como en annealing
ax.set_xlabel('T (temperatura)')
ax.set_ylabel('ξ (longitud de correlación)')
ax.set_title('Test 8: longitud de correlación durante annealing')
ax.grid(alpha=0.3)

# === Plot 3: P(k) en cada fase (Test 9) ===
ax = axes[1, 0]
colors = plt.cm.coolwarm(np.linspace(0, 1, len(phase_results)))
for i, (pd_, color) in enumerate(zip(P_data_per_phase, colors)):
    if pd_['k_centers'] is not None and pd_['valid'] is not None and pd_['valid'].sum() > 0:
        v = pd_['valid']
        ax.plot(pd_['k_centers'][v], pd_['P_avg'][v], 'o-', color=color, alpha=0.8,
                label=f'Fase {i} (T={Ts_arr[i]:.1f}, n_s={ns_per_phase[i]:+.2f})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('k (∝ ω)')
ax.set_ylabel('P(k)')
ax.set_title('Test 9: espectro P(k) por fase del annealing')
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which='both')

# === Plot 4: n_s vs T (Test 9 evolución) ===
ax = axes[1, 1]
ax.plot(Ts_arr, ns_per_phase, 's-', markersize=12, linewidth=2, color='C2')
ax.invert_xaxis()
ax.axhline(0.96, ls='--', color='green', label='Inflación slow-roll (Planck): 0.96')
ax.axhline(1.0, ls=':', color='blue', label='Harrison-Zeldovich: 1.0')
ax.axhline(-1.0, ls=':', color='orange', label='Equipartición acústica: -1.0')
ax.set_xlabel('T (temperatura)')
ax.set_ylabel('n_s')
ax.set_title('Test 9: evolución de n_s durante annealing')
ax.legend(fontsize=9, loc='best')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plots_calibracion_v5_v2/bloque2_inflacion.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot Bloque 2 guardado')

In [ ]:
# ============================================================
# Síntesis final
# ============================================================

print('='*72)
print('SÍNTESIS — Notebook A v2: Calibración + Inflación sin inflatón')
print('='*72)

print(f'\n--- BLOQUE 1: CALIBRACIÓN CORREGIDA ---')
print(f'Test 1 (w_def cosmológico):')
print(f'  ρ_def ∝ V^({slope_rho:+.3f})')
print(f'  w_cosmo = {w_cosmo:+.3f} (esperado CDM: 0)')
if abs(w_cosmo) < 0.05:
    print('  → COMPATIBLE con CDM puro')
elif abs(w_cosmo) < 0.15:
    print('  → Marginal (efectos de tamaño finito esperables)')
else:
    print('  → NO compatible')

print(f'\nTest 2 (gap espectral):')
print(f'  Δω_local (intra-banda) a N=1331: {summary["delta_omega_local_mean"][np.where(np.array(Ns_sorted)==1331)[0][0]]:.4f}')
print(f'  Δω_bulk (al continuo)  a N=1331: {summary["delta_omega_bulk_mean"][np.where(np.array(Ns_sorted)==1331)[0][0]]:.4f}')

print(f'\nTest 3 (ω_def):')
for i, N in enumerate(Ns_sorted):
    print(f'  N={N}: ω_def = {summary["omega_def_mean"][i]:.4f}')
print(f'  Pendiente log-log: ω_def ∝ N^({slope_om:+.3f})')

print(f'\n--- BLOQUE 2: INFLACIÓN SIN INFLATÓN ---')
print(f'Test 7 (suavizado): {n_efolds_eff:.2f} e-folds efectivos durante annealing')
if not np.all(np.isnan(xis_per_phase)):
    print(f'Test 8 (correlación): ξ inicial = {xis_per_phase[0]:.4f}, ξ final = {xis_per_phase[-1]:.4f}')
print(f'Test 9 (n_s): inicial = {ns_per_phase[0]:+.4f}, final = {ns_per_phase[-1]:+.4f}')

print(f'\n--- INTERPRETACIÓN ---')
print(f'(1) El sustrato SÍ produce suavizado significativo durante annealing.')
print(f'(2) La distancia entre n_s clásico (sin inflación) y n_s observado:')
if not np.isnan(ns_per_phase[-1]):
    print(f'    n_s_DEE = {ns_per_phase[-1]:+.3f}, observado = +0.96, diferencia = {0.96-ns_per_phase[-1]:.2f}')
print(f'(3) La calibración Z_φ se sostiene SI w_cosmo < 0.05 (CDM compatible).')

# === Guardar resultados ===
all_results = {
    'calib': {
        'results': calib_results,
        'summary': summary,
        'w_cosmo': w_cosmo,
        'slope_rho': slope_rho,
        'slope_omega': slope_om,
    },
    'inflation': {
        'phase_results': phase_results,
        'vars_per_phase': vars_per_phase.tolist(),
        'xis_per_phase': xis_per_phase.tolist(),
        'ns_per_phase': ns_per_phase.tolist(),
        'n_efolds_eff': n_efolds_eff,
        'T_schedule': T_schedule,
    },
}

with open('plots_calibracion_v5_v2/results_raw.pkl', 'wb') as f:
    pickle.dump(all_results, f)

with open('plots_calibracion_v5_v2/summary.txt', 'w') as f:
    f.write('Calibración corregida + Inflación sin inflatón\n')
    f.write('='*60 + '\n\n')
    f.write('--- BLOQUE 1: CALIBRACIÓN ---\n')
    for r in calib_results:
        f.write(f'N={r["N"]}, seed={r["seed"]}: ω_def={r["omega_def"]:.4f}, '
                f'Δω_local={r["delta_omega_local"]:.4f}, '
                f'Δω_bulk={r["delta_omega_bulk"]:.4f}, '
                f'ρ_def={r["rho_def"]:.6f}\n')
    f.write(f'\nρ_def ∝ V^({slope_rho:+.3f}) → w_cosmo = {w_cosmo:+.3f}\n')
    f.write(f'\n--- BLOQUE 2: INFLACIÓN ---\n')
    for i, T in enumerate(T_schedule):
        f.write(f'Fase {i} (T={T:.2f}): var={vars_per_phase[i]:.4e}, '
                f'ξ={xis_per_phase[i]:.4f}, n_s={ns_per_phase[i]:+.4f}\n')
    f.write(f'\ne-folds efectivos: {n_efolds_eff:.2f}\n')

print('\nResultados guardados en plots_calibracion_v5_v2/')